# MAIZ_14A · Análisis que cierran comentarios parciales · GPU L4 (solo inferencia)

**Entorno: GPU L4** (Entorno de ejecución → Cambiar tipo → L4, RAM alta). Pulse **Ejecutar todas**.

No entrena ni cambia ningún modelo. Descarga los 30 checkpoints públicos de Zenodo (10.5281/zenodo.24064539), verifica su SHA-256 y calcula salidas nuevas, marcadas como análisis de sensibilidad posteriores a la revisión.

| Bloque | Comentario que responde | Qué produce |
|---|---|---|
| Compuerta | — | Concordancia de la nueva inferencia con las predicciones archivadas (PlantVillage y TOM2024, 30 corridas) |
| PlantVillage cerrado | R2 mayor 6 | Macro-F1 con argmax restringido a tizón/mancha, junto al abierto de nueve salidas |
| PlantVillage 3 y 4 clases | R2 mayor 6, R2 menor 8 | Cohorte con roya (1.192) y con sanas que no coinciden con el entrenamiento |
| TOM2024 4 y 5 clases | R2 mayor 6 | Helminthosporiosis-D → leaf_blight; Curvularia-D → leaf_spot (sensibilidad taxonómica) |
| Familias DINOv2 | R2 mayor 4 y 5, Editor 8 | Familias de imágenes casi idénticas y bootstrap por familia de la estabilidad del ranking |
| Grad-CAM 6 arquitecturas | R1 9 | Mapas de las seis arquitecturas (semilla 17) en 16 casos de roya TOM2024 |
| Coincidencia PV sanas | R2 mayor 10, Editor 8 | Identifica las imágenes sanas de PlantVillage que coinciden con el corpus interno |
| Tanzania (opcional) | R2 mayor 10 | Compara el dataset Mduma & Mayo (Mendeley fkw49mz3xs) con las imágenes internas MLN/sanas/MSV |

- **Tiempo aproximado:** 1–1,5 h, unas 3 unidades. La parte más lenta es leer imágenes de Drive y descargar los checkpoints.
- **Reanudación:** si se corta, pulse **Ejecutar todas** otra vez; cada bloque terminado se omite.
- **Resultado:** `MAIZ_14A_Resultados.zip` en `REVISION_POST_REVIEW_20261029/` y como descarga. Ese ZIP es el que debe enviarme.

In [ ]:
import shutil,subprocess
if not shutil.which('nvidia-smi'):raise RuntimeError('Este archivo requiere GPU. Selecciónela en Entorno de ejecución > Cambiar tipo.')
p=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
GPU_NAME=p.stdout.strip();print('GPU:',GPU_NAME)
if 'L4' not in GPU_NAME:print('AVISO: no es L4. Puede continuar; la compuerta de concordancia documentará cualquier diferencia.')

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/TesisQ1_Maiz')
if not ROOT.is_dir() or not (ROOT/'01_datasets').is_dir():
    raise FileNotFoundError('No se encuentra TesisQ1_Maiz/01_datasets en Mi unidad.')
for need in ['REVISION_POST_REVIEW_20261029/evaluation_5seeds/prepared/tom2024_primary/images.tar',
             'REVISION_POST_REVIEW_20261029/training_data_preflight.json',
             '01_datasets/TOM2024/CategoryA_English/CATA-English']:
    print(('OK   ' if (ROOT/need).exists() else 'FALTA'),need)

In [ ]:
# Paquetes: no se reinstala torch. timm se fija en la versión usada para entrenar.
import importlib.metadata as md, subprocess, sys
def ver(p):
    try:return md.version(p)
    except md.PackageNotFoundError:return None
pk=[]
if ver('timm')!='1.0.15':pk.append('timm==1.0.15')
if ver('grad-cam') is None:pk.append('grad-cam==1.5.4')
if ver('ImageHash') is None:pk.append('imagehash==4.3.1')
if pk:subprocess.run([sys.executable,'-m','pip','install','-q',*pk],check=True)
import torch,torchvision,timm
print('torch',torch.__version__,'| torchvision',torchvision.__version__,'| timm',timm.__version__)

In [ ]:
# Código y referencias de MAIZ_14A (verificados por SHA-256)
import base64,hashlib,io,zipfile,importlib,sys
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='c12ff3964318fda96c06d92d9ad396bed26078368da92838ab26d42eab987309'
CODE=Path('/content/MAIZ14_code');CODE.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(raw)).extractall(CODE)
sys.path.insert(0,str(CODE))
import maiz14,maiz14_run as R
importlib.reload(maiz14);importlib.reload(R)
print('Versión:',maiz14.VERSION,'| archivos:',sorted(p.name for p in CODE.iterdir()))

In [ ]:
# Ejecución completa (reanudable). Para omitir la comprobación de Tanzania: tanzania=False
R.run_all(ROOT,tanzania=True)

In [ ]:
# Resumen para revisar en pantalla
import pandas as pd,json
O=ROOT/R.OUT_REL;pd.set_option('display.width',220);pd.set_option('display.max_columns',30)
g=pd.read_csv(O/'agreement_gate.csv');print('Compuerta:',g.gate.value_counts().to_dict(),'| máx. discrepancias:',g.argmax_disagreements.max())
print(json.dumps(json.load(open(O/'cohort_report.json')),indent=1,ensure_ascii=False)[:3000])
print(json.load(open(O/'pv_healthy_internal_overlap_summary.json')))
display(pd.read_csv(O/'analysis/selection_summary.csv'))
display(pd.read_csv(O/'analysis/dinov2_family_counts.csv'))
if (O/'tanzania_check.json').is_file():print(json.load(open(O/'tanzania_check.json')))

In [ ]:
from google.colab import files
files.download('/content/MAIZ_14A_Resultados.zip')